In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The function `range` takes two arguments:
- `start` is an integer,
- `stop` is an integer.

It returns the list $[\texttt{start}, \cdots, \texttt{stop}]$.

In [ ]:
let range (start : int) (stop : int) : int list =
  List.init (max 0 (stop - start + 1)) (fun i -> start + i)

# The 8-Queens Problem

The <a href="https://en.wikipedia.org/wiki/Eight_queens_puzzle">eight queens puzzle</a> is the problem of placing eight chess queens on a chessboard so that no two queens can capture each other.  In <a href="https://en.wikipedia.org/wiki/Chess">chess</a> a queen can capture another piece if this piece is either
<ol>
    <li>in the same row,</li>
    <li>in the same column, or</li>
    <li>in the same diagonal.</li>
</ol>
The image below shows a queen in row 3, column 4.  All the locations where a piece can be captured by this queen are marked with an arrow.

<img src="queen-captures.png">

We will solve this puzzle by encoding it as a *constraint satisfaction problem*.
- The set of values is $\{1,\cdots,8\}$.  These values are interpreted as columns.
- We will use 8 variables $Q_1$, $\cdots$, $Q_8$.  If $Q_r = c$, then the interpretation
  is that there is a queen at the position $(r, c)$, i.e. the queen in row $r$ is
  placed in column $c$.  In OCaml, the variables are written as `q1`, $\cdots$, `q8`.
- The condition
  $$ \bigl\{ \texttt{Q}_i \not= \texttt{Q}_j \;\bigm|\;
      i \in \{1,\cdots,8\} \wedge j \in \{1,\cdots,8\} \wedge j < i \bigr\}
  $$
  specifies that there is at most one queen in a given column.
- The condition
  $$ \bigl\{ |\texttt{Q}_i - \texttt{Q}_j| \not= |i - j| \;\bigm|\;
     i \in \{1,\cdots,8\} \wedge j \in \{1,\cdots,8\} \wedge j < i \bigr\}
  $$
  specifies that there is at most one queen in a given diagonal.

In [ ]:
import_notebook "02-Backtracking-Constraint-Solver.ipynb"

The function `queens_csp` takes one argument:
- `n` is the size of the board.  It is an optional argument with the default value 8.

It returns a *constraint satisfaction problem* that encodes the *n-queens problem*.  As OCaml does not allow a function whose only arguments are optional, `queens_csp` takes the additional argument `()`.

**Implementation:** For all pairs `i < j`, `List.concat_map` collects the two constraints stating that the queens in the rows `i` and `j` are neither in the same column nor in the same diagonal.  The function `abs` computes the absolute value of an integer.

In [ ]:
let queens_csp ?(n = 8) () : csp =
  let vars    = List.map (fun i -> Printf.sprintf "q%d" i) (range 1 n) in
  let values  = range 1 n in
  let constrs =
    List.concat_map
      (fun i ->
         List.concat_map
           (fun j -> [Printf.sprintf "q%d <> q%d" i j;
                      Printf.sprintf "abs (q%d - q%d) <> %d" i j (j - i)])
           (range (i + 1) n))
      (range 1 n)
  in
  { variables = vars; values = values; constraints = constrs }

Next, we create a *CSP* that encodes the *8-queens-puzzle*.

In [ ]:
let csp = queens_csp ()

This *CSP* has 8 variables and 56 constraints.  When we solved the same problem using propositional logic and the *Davis-Putnam algorithm*, we had used 64 propositional variables and 512 clauses.

In [ ]:
(List.length csp.variables, List.length csp.constraints)

The time includes the compilation of the 56 constraints by the OCaml toplevel.

In [ ]:
let solution = time "solution" (fun () -> solve csp)

The function `show_solution` takes three arguments:
* `solution` is a map that contains a variable assignment representing a solution to the n-queens puzzle.  If `solution` maps the variable $\texttt{q}_i$ to the number $k$, then the queen in row $i$ is placed in column $k$.
* `n` specifies the size of the board.  It is an optional argument with the default value 8.
* `width` specifies the display width of the board.  It is an optional argument with the default value `"50%"`.

It displays this solution on a chess board.

In [ ]:
#require "jupyter.notebook";;

In [ ]:
let show_solution ?(n = 8) ?(width = "50%") (solution : assignment) : unit =
  let board = Array.make_matrix n n "" in
  for col = 1 to n do
    match RMap.find_opt (Printf.sprintf "q%d" col) solution with
    | Some row when row <= n -> board.(row - 1).(col - 1) <- "♕"
    | _ -> ()
  done;
  let html = Buffer.create 4096 in
  Buffer.add_string html
    (Printf.sprintf "<div style=\"display: grid; grid-template-columns: repeat(%d, 1fr); width: %s; aspect-ratio: 1/1; border: 2px solid #8b4513;\">" n width);
  for row = 0 to n - 1 do
    for col = 0 to n - 1 do
      let bg_color = if (row + col) mod 2 = 0 then "#ffce9e" else "#d18b47" in
      let cell_content =
        if board.(row).(col) = "" then ""
        else Printf.sprintf "<svg viewBox=\"0 0 100 100\" style=\"width: 80%%; height: 80%%; display: block;\"><text x=\"50%%\" y=\"55%%\" font-size=\"90\" text-anchor=\"middle\" dominant-baseline=\"middle\" fill=\"black\">%s</text></svg>" board.(row).(col)
      in
      Buffer.add_string html
        (Printf.sprintf "<div style=\"display: flex; align-items: center; justify-content: center; background-color: %s; overflow: hidden;\">%s</div>" bg_color cell_content)
    done
  done;
  Buffer.add_string html "</div>";
  ignore (Jupyter_notebook.display "text/html" (Buffer.contents html))

In [ ]:
match solution with
| Some s -> show_solution s
| None   -> print_endline "No solution found."

If we would use a brute force approach that checks all permutations, we would have to check $8! = 40\,320$ different configurations.  If we would check all assignments, we would have to check $8^8$ assignments:

In [ ]:
8. ** 8.